# CIMTinyTO: Micro-ResNet Edge AI Vision Demonstration (Pillar 6.1)

**Hardware Target:** AMD Xilinx Zynq-7020 (PYNQ-Z2) FPGA Emulation & SkyWater 130nm ASIC Core
**Architecture:** $16 \times 16$ Stochastic Compute-in-Memory (SCIM) Macro (Mode 1: Hybrid ReLU)

---

## Silicon Realities & Microarchitectural Principles
1. **Mode 1 Hybrid ReLU Arithmetic:** Bipolar weights ($W \in \{-1, +1\}$) and unipolar activations ($A \in [0, 255]$) are multiplied in-place using single standard-cell AND gates and 4:2 Wallace tree compressors.
2. **Stationary Standard-Cell DFF Memory:** 256 standard-cell flip-flops store the $16 \times 16$ weight tile on-chip. Shifting occurs **ONCE per layer**, slashing dynamic weight switching power ($P = C V^2 f = 0$) during spatial inference.
3. **Multi-Tier Memory Wall Benchmark:** Quantifies the software-to-hardware transition:
   - **Tier 1:** Pure Python MMIO (~180 us/tile -> ~0.7 FPS)
   - **Tier 2:** Compiled C MMIO via direct pointers (~15 us/tile -> ~8.3 FPS)
   - **Tier 3:** Autonomous Hardware Feeder / AXI-DMA (~6.08 us/tile -> 21.3 - 45 FPS)


In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Configure paths to micro_resnet module
CURRENT_DIR = os.getcwd()
PROJ_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "../../.."))
for p in [os.path.join(CURRENT_DIR, "../micro_resnet"), PROJ_ROOT]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from fpga.pynq_z2.micro_resnet.vision_pipeline import EdgeVisionPipeline
from fpga.pynq_z2.micro_resnet.micro_resnet_model import MicroResNet16

print("Initializing Edge Vision Pipeline...")
# Connect to physical hardware if on PYNQ Linux, or engage bit-exact host emulation
pipeline = EdgeVisionPipeline(force_emulation=False, seed=42)
summary = pipeline.tier1_engine.model.get_summary()
print(f"Model: {summary['model_name']} | Weight Footprint: {summary['total_weight_bytes']} Bytes ({summary['total_weight_bits']} bits)")
print(f"Total 16x16 Tiles per Frame: {summary['tiles_per_frame']['total']}")


## 1. Input Image Ingestion & Synthetic Test Patterns
We generate deterministic visual benchmark patterns to inspect intermediate spatial feature maps.

In [ ]:
patterns = ["checkerboard", "circle", "horizontal_stripes", "gradient"]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, p in zip(axes, patterns):
    img_pat = pipeline.generate_synthetic_pattern(p)
    ax.imshow(img_pat)
    ax.set_title(p.capitalize())
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Multi-Tier Micro-ResNet Inference Benchmark
Run full inference across Tier 1 (Python MMIO), Tier 2 (Compiled C MMIO), and Tier 3 (Line-Rate DMA).

In [ ]:
test_img = pipeline.generate_synthetic_pattern("circle")

print(">>> Running Tier 1 (Pure Python MMIO)...")
res1 = pipeline.run_inference(test_img, tier=1)

print(">>> Running Tier 2 (Compiled C MMIO)...")
res2 = pipeline.run_inference(test_img, tier=2)

print(">>> Running Tier 3 (Autonomous Feeder / AXI-DMA Model)...")
res3 = pipeline.run_inference(test_img, tier=3)

# Display scorecard
print("\n" + "=" * 75)
print(f"{'Tier':<30} | {'FPS':<10} | {'Tile Latency':<15} | {'Prediction'}")
print("-" * 75)
for r in [res1, res2, res3]:
    print(f"{r['tier']:<30} | {r['fps']:>6.2f} FPS | {r['tile_latency_us']:>8.2f} us/tile | {r['pred_class'].upper()}")
print("=" * 75)


## 3. Real-Time Heads-Up Display (HUD) Overlay
Annotates live camera/video frames with top-1 prediction, confidence score, and real-time FPS telemetry.

In [ ]:
hud_image = pipeline.render_hud_overlay(res2, scale=8)
plt.figure(figsize=(6, 8))
plt.imshow(hud_image)
plt.title("Edge Vision HUD Overlay (HDMI / Camera)", fontsize=12)
plt.axis("off")
plt.show()


## 4. Stationary Weight Power Analysis & Silicon Footprint
Quantifying how stationary standard-cell DFF memory slashes memory traffic.

In [ ]:
total_tiles = res2['total_tiles']
weight_loads = res2['weight_loads']
redundant_shifts_avoided = (total_tiles - weight_loads) * 256
saving_pct = (1.0 - weight_loads / total_tiles) * 100.0

print(f"Total MVM Tiles Computed:            {total_tiles}")
print(f"Weight Matrix Reloads:                {weight_loads} (1 load per layer)")
print(f"Redundant Shift Cycles Saved:         {redundant_shifts_avoided:,} cycles")
print(f"Dynamic Weight Switching Suppression: {saving_pct:.2f}% power reduction!")
